# Day 2 — Baseline + XGBoost

Goal: a trained, honestly-evaluated classifier with documented numbers, saved to `../src/model.joblib`.

Run top to bottom with **Shift+Enter**. Pick your **Python** kernel if asked.
Read curriculum sections 2–7 first — especially the evaluation section.

## Step 1 — load the Day 1 dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/trials_features.csv")
df["label"] = (df["overall_status"].str.upper() == "COMPLETED").astype(int)
print("Rows:", len(df))
print("Completion rate:", round(df["label"].mean(), 3))
df.head()

## Step 2 — define features and split

We drop `brief_summary` (that text is for Day 3 retrieval, not this tabular model) and `overall_status` (it IS the label — keeping it would be leakage).

In [ ]:
from sklearn.model_selection import train_test_split

num = ["enrollment", "number_of_arms", "start_year", "duration_months",
       "n_conditions", "n_interventions"]
cat = ["phase", "sponsor_class", "allocation", "intervention_model",
       "masking", "primary_purpose"]

X = df[num + cat]
y = df["label"]
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", Xtr.shape, " Test:", Xte.shape)

## Step 3 — preprocessing pipeline

Imputation + scaling for numerics, imputation + one-hot for categoricals. Fitting inside a pipeline prevents leakage (transforms learn from train only).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
])

## Step 4 — baseline: logistic regression

Always report a simple baseline so you can prove the fancy model earned its complexity.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

logit = Pipeline([("pre", pre),
                  ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))]).fit(Xtr, ytr)
print("Logistic Regression AUC:", round(roc_auc_score(yte, logit.predict_proba(Xte)[:, 1]), 4))

## Step 5 — main model: XGBoost (with imbalance handling)

`scale_pos_weight` counteracts class imbalance so the model doesn't ignore the minority class (the failures). Value = (#negatives / #positives) on the training set.

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score

spw = (ytr == 0).sum() / (ytr == 1).sum()   # auto-computed from your data
print("scale_pos_weight:", round(spw, 3))

model = Pipeline([("pre", pre),
    ("clf", xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw, eval_metric="auc"))]).fit(Xtr, ytr)

proba = model.predict_proba(Xte)[:, 1]
print("\nXGBoost AUC:", round(roc_auc_score(yte, proba), 4))
print("\n", classification_report(yte, (proba > 0.5).astype(int)))

## Step 6 — calibration check (high-signal extra)

Does "70% predicted" actually complete ~70% of the time? A model can rank well (good AUC) yet report poorly-calibrated probabilities.

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

frac_pos, mean_pred = calibration_curve(yte, proba, n_bins=10)
plt.plot(mean_pred, frac_pos, marker="o", label="model")
plt.plot([0, 1], [0, 1], "--", label="perfect")
plt.xlabel("Predicted probability"); plt.ylabel("Actual completion rate")
plt.title("Calibration"); plt.legend(); plt.show()

## Step 7 — feature importance with SHAP (gold for the CVS 'explain to stakeholders' angle)

In [ ]:
import shap

# transform a sample through the pipeline, then explain the XGBoost step
X_sample = Xte.sample(min(2000, len(Xte)), random_state=42)
X_trans = model.named_steps["pre"].transform(X_sample)
feat_names = model.named_steps["pre"].get_feature_names_out()

explainer = shap.TreeExplainer(model.named_steps["clf"])
shap_values = explainer.shap_values(X_trans)
shap.summary_plot(shap_values, X_trans, feature_names=feat_names, max_display=15)

## Step 8 — save the model for the API (Day 4 needs this)

In [ ]:
import joblib
joblib.dump(model, "../src/model.joblib")
print("Saved ../src/model.joblib")